
# MVP Sprint 6 — Detección de Phishing

Prototipo desplegado con **Voila en AWS**.

Este tablero muestra:

- Predicción de URLs con el modelo final.
- Umbral recomendado de negocio.
- KPIs técnicos y económicos.
- Gráficos de Business Value.
- Trazabilidad MLflow.
- Estado de persistencia MongoDB.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, Image, clear_output

ROOT = Path.cwd()
MODEL_PATH = ROOT / "models" / "final_model.pkl"
TEST_PATH = ROOT / "data" / "processed" / "test.csv"
BV_PATH = ROOT / "reports" / "business_value_summary.csv"
MLFLOW_PATH = ROOT / "reports" / "mlflow_tracking_summary.csv"
MONGO_PATH = ROOT / "reports" / "mongodb_export_summary.csv"
FIG_DIR = ROOT / "reports" / "figures"

def load_csv(path):
    return pd.read_csv(path) if path.exists() else None

def get_business_value():
    df = load_csv(BV_PATH)
    if df is None:
        return {}, 0.09
    d = dict(zip(df["item"], df["value"]))
    return d, float(d.get("umbral_recomendado", 0.09))

business, threshold_default = get_business_value()

try:
    model = joblib.load(MODEL_PATH)
    model_status = "loaded"
    model_error = ""
except Exception as exc:
    model = None
    model_status = "error"
    model_error = str(exc)

display(Markdown(f'''
## Estado del modelo

| Elemento | Valor |
|---|---|
| Modelo | `{MODEL_PATH}` |
| Estado | **{model_status}** |
| Umbral recomendado | **{threshold_default}** |
'''))

if model_error:
    display(Markdown(f"**Error:** `{model_error}`"))


In [ ]:

display(Markdown("## Resumen Ejecutivo y Business Value"))

def val(k, default=""):
    return business.get(k, default)

kpis = pd.DataFrame([
    ["KPI económico principal", val("kpi_economico_principal", "ahorro_neto_esperado")],
    ["KPI técnico soporte", val("kpi_tecnico_soporte", "recall_phishing")],
    ["Umbral recomendado", val("umbral_recomendado", threshold_default)],
    ["Recall recomendado", val("recall_recomendado", "")],
    ["FN recomendado", val("fn_recomendado", "")],
    ["FP recomendado", val("fp_recomendado", "")],
    ["Ahorro neto recomendado USD", val("ahorro_neto_recomendado_usd", "")],
    ["ROI recomendado", val("roi_recomendado", "")],
    ["Valor por 1000 URLs USD", val("valor_por_1000_urls_recomendado_usd", "")],
    ["Ahorro incremental vs 0.50 USD", val("ahorro_incremental_vs_050_usd", "")]
], columns=["Indicador", "Valor"])

display(kpis)


In [ ]:

display(Markdown("## Gráficos de resultados y valor de negocio"))

figs = [
    "business_value_thresholds.png",
    "business_value_scenarios.png",
    "final_comparison.png",
    "gain_curve.png",
    "ensemble_comparison.png",
    "tuning_comparison.png",
    "final_validation_curves.png"
]

shown = 0
for f in figs:
    p = FIG_DIR / f
    if p.exists():
        display(Markdown(f"### {f}"))
        display(Image(filename=str(p)))
        shown += 1

if shown == 0:
    display(Markdown("No se encontraron gráficos en `reports/figures`."))


In [ ]:

display(Markdown("## Predicción Operativa"))

uploader = widgets.FileUpload(accept=".csv", multiple=False, description="Cargar CSV")
btn_test = widgets.Button(description="Usar test.csv", button_style="info")
btn_predict = widgets.Button(description="Predecir", button_style="success")
threshold_box = widgets.FloatText(value=float(threshold_default), description="Threshold")
out = widgets.Output()
current = {"df": None}

def read_upload():
    if not uploader.value:
        return None
    value = uploader.value
    import io
    if isinstance(value, dict):
        item = next(iter(value.values()))
        return pd.read_csv(io.BytesIO(item["content"]))
    item = value[0]
    return pd.read_csv(io.BytesIO(item["content"]))

def expected_features():
    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)
    return None

def prepare_x(df):
    x = df.drop(columns=["Result", "target", "prediction", "prediction_label", "phishing_probability"], errors="ignore")
    exp = expected_features()
    if exp:
        missing = [c for c in exp if c not in x.columns]
        if missing:
            raise ValueError(f"Faltan columnas requeridas: {missing[:10]}")
        x = x[exp]
    return x

def predict(df, thr):
    if model is None:
        raise RuntimeError("Modelo no cargado.")
    x = prepare_x(df)
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(x)[:, 1]
        pred = (prob >= thr).astype(int)
    else:
        pred = model.predict(x)
        prob = np.where(pred == 1, 1.0, 0.0)
    res = df.copy()
    res["phishing_probability"] = prob
    res["prediction"] = pred
    res["prediction_label"] = np.where(pred == 1, "phishing", "legitimate")
    return res

def use_test(_):
    with out:
        clear_output()
        if TEST_PATH.exists():
            df = pd.read_csv(TEST_PATH)
            current["df"] = df
            display(Markdown(f"Dataset cargado: `{TEST_PATH}`"))
            display(df.head())
        else:
            display(Markdown("No existe `data/processed/test.csv`."))

def run_predict(_):
    with out:
        clear_output()
        try:
            df = read_upload()
            if df is None:
                df = current["df"]
            if df is None:
                display(Markdown("Carga un CSV o usa test.csv."))
                return
            res = predict(df, float(threshold_box.value))
            n = len(res)
            ph = int((res["prediction_label"] == "phishing").sum())
            leg = int((res["prediction_label"] == "legitimate").sum())
            avg = float(res["phishing_probability"].mean())
            display(Markdown("### KPIs de predicción"))
            display(pd.DataFrame([
                ["URLs evaluadas", n],
                ["Phishing detectados", ph],
                ["Legítimas", leg],
                ["Probabilidad promedio phishing", round(avg, 4)],
                ["Threshold usado", threshold_box.value]
            ], columns=["Indicador", "Valor"]))
            display(Markdown("### Primeras predicciones"))
            display(res[["prediction", "prediction_label", "phishing_probability"]].head(30))
        except Exception as exc:
            display(Markdown(f"**Error:** `{exc}`"))

btn_test.on_click(use_test)
btn_predict.on_click(run_predict)

display(widgets.VBox([
    widgets.HTML("<b>Cargar datos y ejecutar predicción</b>"),
    uploader,
    threshold_box,
    widgets.HBox([btn_test, btn_predict]),
    out
]))


In [ ]:

display(Markdown("## Resumen MLflow"))

ml = load_csv(MLFLOW_PATH)
if ml is not None:
    display(pd.DataFrame([
        ["Runs registrados", len(ml)],
        ["Success", int((ml["status"] == "success").sum()) if "status" in ml.columns else ""],
        ["Partial", int((ml["status"] == "partial").sum()) if "status" in ml.columns else ""],
        ["Failed", int((ml["status"] == "failed").sum()) if "status" in ml.columns else ""]
    ], columns=["Métrica", "Valor"]))
    cols = [c for c in ["model_name", "run_type", "sprint", "status", "metrics_logged"] if c in ml.columns]
    display(ml[cols])
else:
    display(Markdown("No se encontró `reports/mlflow_tracking_summary.csv`."))

display(Markdown("## Estado MongoDB"))

mg = load_csv(MONGO_PATH)
if mg is not None:
    display(mg)
else:
    display(Markdown("MongoDB es opcional. No se encontró reporte de exportación."))
